# 2-fish recording: better list of good frames (mask length ≥ 0.8 × median)

As `EOD_QC_length_limit_3fish.ipynb`, for the 2-fish recording. The mask QC of `EOD_assignment_2_fish_silent.ipynb` keeps a fish's
mask when its principal-axis length is 0.7–1.2 × that fish's median. The **better list** keeps the common good frames
(`results/eod_ground_truth/common_good_frames.csv`: mask QC and SLEAP both good) where every fish's mask is at least **0.8** × that
fish's median length, saved as `results/eod_ground_truth/good_frames_better_list.csv` (same format). Lengths and medians as in the QC
(`mask_metrics.shape_metrics`, median over all frames where the fish has a mask). Emitter: fish 1.

In [1]:
import pickle
import numpy as np
import pandas as pd
import mask_metrics as mm
%matplotlib inline

TRACK_PKL = 'data/2_fish_silent/elephantnose_20240213T143321_tracking_final.pkl'
OUT = 'results/eod_ground_truth/conductivities'
FISH, EMITTER = (0, 1), 1
LENGTH_MIN_OLD, LENGTH_MIN_NEW = 0.7, 0.8

with open(TRACK_PKL, 'rb') as f:
    frames = pickle.load(f)['frames']
rows = []
for k, fr in frames.items():
    for l, p, b in zip(fr['labels'], fr['masks'], fr['boxes']):
        if l in FISH: rows.append(dict(frame=k, fish=l, **mm.shape_metrics(mm.unpack_crop(p, b), b)))
q = pd.DataFrame(rows)
q['length_ratio'] = q.major / q.groupby('fish').major.transform('median')
L = q.pivot_table(index='frame', columns='fish', values='length_ratio', aggfunc='min')   # (frame, fish) length / median
print('median principal-axis length (px):', q.groupby('fish').major.median().round(1).to_dict())

median principal-axis length (px): {0: 163.0, 1: 222.9}


In [2]:
A = pd.read_csv(f'{OUT}/eod_assignments.csv', index_col='row')
S = pd.read_csv('results/eod_assignment_2_fish_silent/eod_assignment_summary.csv')      # Denis' run, incl. the mask-midline run
assert (S.Frame.to_numpy() == A.Frame.to_numpy()).all()
A["fish | Denis' pipeline with my mask midline"] = S.discharging_fish_mask_midline.map({'A': 1, 'B': 0}).to_numpy()
pipes = [c for c in A.columns if c.startswith('fish | ')]
lr = L.reindex(A.Frame.to_numpy()); lr.index = A.index
common = pd.read_csv('results/eod_ground_truth/common_good_frames.csv').frame
good = A.Frame.isin(common).to_numpy()
assert (lr[good].min(axis=1) >= LENGTH_MIN_OLD - 1e-9).all()          # the common good frames already pass 0.7 for every fish
better = good & (lr.min(axis=1) >= LENGTH_MIN_NEW).to_numpy()
better_frames = np.sort(A.Frame[better].unique())
pd.Series(better_frames, name='frame').to_csv('results/eod_ground_truth/good_frames_better_list.csv', index=False)
print(f'good_frames_better_list.csv: {len(better_frames)} of {len(common)} common good frames '
      f'({good.sum() - better.sum()} of {good.sum()} EODs dropped); '
      f'fish {EMITTER} limit only would keep {(good & (lr[EMITTER] >= LENGTH_MIN_NEW).to_numpy()).sum()} EODs')
SETS = {'good frames': good, 'better list': better, 'dropped': good & ~better}
acc = pd.DataFrame({f'{s} (n = {m.sum()})': [100 * (A.loc[m, c] == EMITTER).mean() for c in pipes] for s, m in SETS.items()},
                   index=[c.replace('fish | ', '') for c in pipes]).rename_axis(f'% correct (fish {EMITTER})')
acc.to_csv(f'{OUT}/accuracy_length_limit_{LENGTH_MIN_NEW}.csv')
display(acc.round(2))

good_frames_better_list.csv: 3500 of 3531 common good frames (31 of 3532 EODs dropped); fish 1 limit only would keep 3514 EODs


,good frames (n = 3532),better list (n = 3501),dropped (n = 31)
% correct (fish 1),,,
"previous σ (interior 200, skin 0.05, 0.2 cm), sink at tail tip",96.60,96.72,83.87
"previous σ (interior 200, skin 0.05, 0.2 cm), sink 5% from tail tip",95.27,95.32,90.32
"BEM σ (interior 100, skin 0.05, 0.005 cm), sink at tail tip",96.63,96.74,83.87
"BEM σ (interior 100, skin 0.05, 0.005 cm), sink 5% from tail tip",95.16,95.26,83.87
Denis' pipeline (latest run),95.87,95.94,87.10
Denis' pipeline with my mask midline,96.86,96.97,83.87
